# WORD SIMILARITY AND ANOLOGY

In practice, word vectors that are pretrained on large corpora can be applied to downstream natural language processing tasks, which will be covered later. 

To demonstrate semantics of pretrained word vectors from large corpora in a straightforward way, let’s apply them in the word similarity and analogy tasks.

In [1]:
import os
import torch
from utils import spy
from torch import nn

## STEP 1 - Loading pretrained word vectors

Below lists pretrained GloVe embeddings of dimension 50, 100, and 300, which can be downloaded from the GloVe website. The pretrained fastText embeddings are available in multiple languages. Here we consider one English version (300-dimensional “wiki.en”) that can be downloaded from the fastText website.

To load these pretrained GloVe and fastText embeddings, we define the following TokenEmbedding class.

In [37]:
class TokenEmbedding:
    def __init__(self, embedding_name):
        self.idx_to_token, self.idx_to_vec = self._load_embedding(embedding_name)
        self.unknown_idx = 0
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}


    def _load_embedding(self, embedding_name):
        idx_to_token, idx_to_vec = ['<unk>'], []
        data_dir = spy.download_extract(embedding_name, folder='../../data')
        # GloVe website: https://nlp.stanford.edu/projects/glove/
        # fastText website: https://fasttext.cc/
        with open(os.path.join(data_dir, 'vec.txt'), 'r') as f:
            for line in f:
                elems = line.rstrip().split(' ')
                token, elems = elems[0], [float(elem) for elem in elems[1: ]]
                # Skip the header information
                if len(elems) > 1:
                    idx_to_token.append(token)
                    idx_to_vec.append(elems)
        # Vector representation for '<unk>'
        idx_to_vec = [[0] * len(idx_to_vec[0])] + idx_to_vec
        return idx_to_token, torch.tensor(idx_to_vec)
    

    def __getitem__(self, tokens):
        indices = [self.token_to_idx.get(token, self.unknown_idx) for token in tokens]
        vecs = self.idx_to_vec[torch.tensor(indices)]
        return vecs
    

    def __len__(self):
        return len(self.idx_to_token)

Below we load the 50-dimensional GloVe embeddings (pretrained on a Wikipedia subset). When creating the TokenEmbedding instance, the specified embedding file has to be downloaded if it was not yet.

In [38]:
glove_6b50d = TokenEmbedding('http://d2l-data.s3-accelerate.amazonaws.com/glove.6B.50d.zip')

Output the vocabulary size. The vocabulary contains 400000 words (tokens) and a special unknown token.

In [39]:
len(glove_6b50d)

400001

We can get the index of a word in the vocabulary, and vice versa.



In [40]:
glove_6b50d.token_to_idx['devine'], glove_6b50d.idx_to_token[24258]

(24258, 'devine')

## STEP 2 - Apply pretrained word vectors

Using the loaded GloVe vectors, we will demonstrate their semantics by applying them in the following word similarity and analogy tasks.

### 2.1. Word Similarity

In order to find semantically similar words for an input word based on cosine similarities between word vectors, we implement the following knn ($k$-nearest neighbors) function.

In [41]:
def knn(W, x, k):
    # Add 1e-9 for numerical stability
    cos = torch.mv(W, x.reshape(-1, )) / (torch.sqrt(torch.sum(W * W, dim=1) + 1e-9) * 
                                          torch.sqrt(torch.sum(x * x)))
    _, topk = torch.topk(cos, k=k)     # This returns values and indices
    return topk, [cos[int(i)] for i in topk]

Then, we search for similar words using the pretrained word vectors from the TokenEmbedding instance embed.

In [42]:
def get_similar_tokens(query_token, k, embed):
    topk, cos = knn(embed.idx_to_vec, embed[[query_token]], k+1)
    for i, c in zip(topk[1: ], cos[1: ]):         # Exclude the input word
        print(f"Cosine sim={float(c):.3f}: {embed.idx_to_token[int(i)]}")

The vocabulary of the pretrained word vectors in `glove_6b50d` contains 400000 words and a special unknown token. Excluding the input word and unknown token, among this vocabulary let’s find three most semantically similar words to word `beautiful`.

In [43]:
get_similar_tokens('beautiful', 3, glove_6b50d)

Cosine sim=0.921: lovely
Cosine sim=0.893: gorgeous
Cosine sim=0.830: wonderful


In [44]:
get_similar_tokens('strong', 3, glove_6b50d)

Cosine sim=0.866: stronger
Cosine sim=0.856: strongest
Cosine sim=0.855: contrast


### 2.2. Word Analogy

Besides finding similar words, we can also apply word vectors to word analogy tasks. For example, “man”:“woman”::“son”:“daughter” is the form of a word analogy: “man” is to “woman” as “son” is to “daughter”. 

Specifically, the word analogy completion task can be defined as: for a word analogy $a : b :: c : d$, given the first three words $a, b, c$, find $d$. Denote the vector of word $w$ by $vec(w)$. To complete the analogy, we will find the word whose vector is most similar to the result of $vec(c) +vec(b) - vec(a)$.

In [45]:
def get_analogy(token_a, token_b, token_c, embed):
    vecs = embed[[token_a, token_b, token_c]]
    x = vecs[1] + vecs[2] - vecs[0]
    topk, cos = knn(embed.idx_to_vec, x, 1)
    return embed.idx_to_token[int(topk[0])]

Let’s verify the “male-female” analogy using the loaded word vectors.

In [46]:
get_analogy('man', 'woman', 'son', glove_6b50d)

'daughter'

In [49]:
get_analogy('beijing', 'china', 'tokyo', glove_6b50d)

'japan'

For the “adjective-superlative adjective” analogy such as “bad”:“worst”::“big”:“biggest”, we can see that the pretrained word vectors may capture the syntactic information.

In [51]:
get_analogy('bad', 'worst', 'big', glove_6b50d)

'biggest'

To show the captured notion of past tense in the pretrained word vectors, we can test the syntax using the “present tense-past tense” analogy: “do”:“did”::“go”:“went”.

In [58]:
get_analogy('do', 'did', 'go', glove_6b50d)

'went'